# **Índice de similitud entre jugadores y equipos (*cosine similarity*)**

En este estudio se calcula un **índice de similitud** entre pares de jugadores (y de equipos) usando ***cosine similarity***. El objetivo es responder a "¿qué jugadores se parecen más a X?" de forma cuantitativa, a partir de sus estadísticas.

La *cosine similarity* es una métrica que mide el parecido entre dos vectores calculando el **coseno del ángulo** que forman: ignora la magnitud y se fija en la **orientación**, es decir, en el *patrón* de juego más que en el volumen absoluto.

---

## **1. Procesado de datos**

El primer bloque construye un *dataframe* con las estadísticas agregadas de cada jugador y su contexto. Los pasos principales:

### **1.1. Creación de métricas derivadas**

A partir de los conteos brutos (pases, tiros, duelos…) se calculan **ratios y porcentajes** interpretables. Para evitar divisiones por cero se usa una división segura `df_safe_div(num, den)` que devuelve `NaN` cuando el denominador es 0 o hay nulos.

Se generan, entre muchas otras:

- **Pase y posesión:** `PassAccuracy = AccuratePasses / Passes`, `ProgressiveFieldTilt`, `PossessionLossRate`, etc.
- **Creación:** `KeyPassesPerPass`, `AssistConversion`, `ExpectedAssistsPerKeyPass`, `CrossAccuracy`…
- **Finalización:** `ShotAccuracy = ShotsOnTarget / TotalShots`, `GoalConversion`, `ExpectedGoalsPerShot`, `GoalsMinusxG`…
- **Duelos y defensa:** `DuelWinRate`, `AerialWinRate`, `TackleAccuracy`, `DefensiveActionSuccess`…
- **Portero:** `SaveRate`, `SavedShotsInsideBoxRate`, `HighClaimRate`, `PenaltySaveRate`…
- **Distribución de pases por zona y dirección** (*shares* y precisiones).

Además se calculan versiones **por 90 minutos** de las columnas de volumen:

$$
\text{stat}_{/90} = \text{stat} \cdot \frac{90}{\text{MinutosJugados}}
$$

### **1.2. Agregación de la temporada**

Las estadísticas por partido se agregan por jugador: las de **volumen se suman** y las de **posición/medias** (coordenadas medias de pase y tiro, Elo…) se promedian. Tras agregar, se recalculan los ratios sobre los totales de temporada.

### **1.3. Limpieza y codificación final**

- Se filtran jugadores con **≥ 500 minutos** y se exige tener `DateBirth`, `Height` y `PrefFoot`.
- Se calcula la **edad** a partir de la fecha de nacimiento.
- Se añade una **posición general** (`Goalkeeper`, `Defender`, `Midfielder`, `Attacker`).
- Los nulos numéricos se imputan con la **media por posición** (más justo que una media global).
- Las variables categóricas (pie preferente, rol, posición general…) se transforman a numéricas mediante ***one-hot encoding***.

In [ ]:
import pandas as pd
import os
import numpy as np
from typing import Tuple
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import warnings
from pandas.errors import PerformanceWarning

warnings.simplefilter(action="ignore", category=PerformanceWarning)

# Configuración
ACT_SEASON = "2526"
DATA_PATH = r"d:\data"
SILVER_DATA_PATH = os.path.join(DATA_PATH, "silver")

# Lectura de dataframes con los que vamos a trabajar
player_info_df = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "player_clean_3.csv"), sep=";")
player_stats_df = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_3", "player_stats_clean_3_2.csv"), sep=";")
match_info_df = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_2", "match_clean_2.csv"), sep=";")

# Lista para ordenar el dataframe en un futuro
list_to_order_df = ["Player","DateBirth","Height","PrefFoot","Position","Role","MinutesPlayed","Matches","Elo","OppElo","Score","OppScore","TouchesPer90","PassesPer90",
                    "AccuratePassesPer90","PassAccuracy","PassesPerTouch","TouchesPerPass","PossessionLostPer90","PossessionLossRate","UnsuccessfulTouchesPer90",
                    "UnsuccessfulTouchRate","DispossessedPer90","DispossessedRate","OwnHalfPassesPer90","AccurateOwnHalfPassesPer90","OwnHalfPassAccuracy","OwnHalfPassShare",
                    "OppositionHalfPassesPer90","AccurateOppositionHalfPassesPer90","OppositionHalfPassAccuracy","OppositionHalfPassShare","LongBallsPer90","AccurateLongBallsPer90",
                    "LongBallAccuracy","LongBallShare","ProgressiveFieldTiltPer90","KeyPassesPer90","KeyPassesPerPass","GoalAssistsPer90","AssistConversion","ExpectedAssistsPer90",
                    "ExpectedAssistsPerKeyPass","ExpectedAssistsPerPass","CrossesPer90","AccurateCrossesPer90","CrossAccuracy","CrossShare","AccurateCrossShare",
                    "BigChancesCreatedPer90","BigChancesMissedPer90","BigChancesPerKeyPass","TotalShotsPer90","ShotsOnTargetPer90","ShotsOffTargetPer90","BlockedShotsPer90",
                    "GoalsPer90","ExpectedGoalsPer90","ExpectedGoalsOnTargetPer90","ShotAccuracy","GoalConversion","BigChanceMissRate","ExpectedGoalsPerShot",
                    "ExpectedGoalsOnTargetPerShotOnTarget","GoalsMinusxGPer90","GoalsMinusxGOTPer90","BlockedShotRate","ShotsOnTargetRate","ShotsOffTargetRate","GoalsPerShotOnTarget",
                    "HitWoodworkPer90","HitWoodworkRate","OffsidesPer90","OffsideRate","ShotMeanX","ShotMeanY","ShotMeanLength","ShotTotalLenghtPer90","DuelsWonPer90",
                    "DuelsLostPer90","DuelWinRate","AerialWonPer90","AerialLostPer90","AerialWinRate",
                    "ContestsPer90","ContestsWonPer90","ContestWinRate","ChallengesLostPer90","ChallengesLostRate","TacklesPer90","TacklesWonPer90","TackleAccuracy",
                    "InterceptionsPer90","InterceptionShare","ClearancesPer90","ClearanceShare","OutfielderBlocksPer90","BlockShare","BallRecoveriesPer90","BallRecoveryRate",
                    "DefensiveActionsPer90","DefensiveActionSuccess","LastManTacklesPer90","LastManTackleRate","ClearanceOffLinePer90","ClearanceOffLineRate","ErrorsLeadToShotPer90",
                    "ErrorsLeadToShotRate","ErrorsLeadToGoalPer90","ErrorsLeadToGoalRate","GoalsConcededPerDefAction","SavesPer90","SaveRate","SavedShotsInsideBoxPer90",
                    "SavedShotsInsideBoxRate","CrossesNotClaimedPer90","CrossesNotClaimedRate","HighClaimsPer90","HighClaimRate","PunchesPer90","PunchRate","KeeperSweeperActionsPer90",
                    "AccurateKeeperSweeperActionsPer90","KeeperSweeperAccuracy","GoalKicksPer90","GoalKicksRate","PenaltyFacedPer90","PenaltySavesPer90","PenaltySaveRate",
                    "PenaltyGoalsConcededPer90","PenaltyGoalConcededRate","GoalsConcededPer90","GoalsPreventedPer90","GoalsPreventedDiffPer90","CleanSheetsPer90","FoulsPer90",
                    "YellowCardsPer90","RedCardsPer90","CardsPerFoul","YellowCardsPerFoul","RedCardsPerFoul","FoulsPerDefAction","WasFouledPer90","WasFouledRate","PenaltyWonPer90",
                    "PenaltyWonRate","PenaltyMissedPer90","PenaltyMissRate","PenaltyConcededPer90","PenaltyConcededRate","OwnGoalsPer90","OwnGoalRate","CornersWonPer90",
                    "CornersWonRate","CornersLostPer90","CornersLostRate","CornersTakenPer90","CornersTakenRate","PassMeanAngle","PassMeanLength","PassTotalLengthPer90",
                    "PassMeanX","PassMeanY","PassPercZoneBack","PassPercZoneCenter","PassPercZoneLeft","PassPercZoneRight","PassShareZoneBack","PassShareZoneCenter",
                    "PassShareZoneLeft","PassShareZoneRight","PassPercDirBackward","PassPercDirBackwardLeft","PassPercDirBackwardRight","PassPercDirForward","PassPercDirForwardLeft",
                    "PassPercDirForwardRight","PassPercDirLeft","PassPercDirRight","PassShareDirBackward","PassShareDirBackwardLeft","PassShareDirBackwardRight","PassShareDirForward",
                    "PassShareDirForwardLeft","PassShareDirForwardRight","PassShareDirLeft","PassShareDirRight"]

# División segura entre columnas de dataframes
def df_safe_div(num, den, ndigits=4):
    result = np.where((den != 0) & (~pd.isna(num)) & (~pd.isna(den)), num / den, np.nan)
    return np.round(result, ndigits)

# Creación de nuevas métricas para el dataframe de jugadores
def player_new_metrics(player_stats_df: pd.DataFrame) -> pd.DataFrame:

    df = player_stats_df.copy()

    # Factor x90
    factor_90 = df_safe_div(90, df["MinutesPlayed"])

    # Pase y posesión
    df["PassAccuracy"] = df_safe_div(df["AccuratePasses"], df["Passes"])
    df["OwnHalfPassAccuracy"] = df_safe_div(df["AccurateOwnHalfPasses"], df["OwnHalfPasses"])        
    df["OppositionHalfPassAccuracy"] = df_safe_div(df["AccurateOppositionHalfPasses"], df["OppositionHalfPasses"]) 
    df["LongBallAccuracy"] = df_safe_div(df["AccurateLongBalls"], df["LongBalls"]) 
    df["OwnHalfPassShare"] = df_safe_div(df["OwnHalfPasses"], df["Passes"]) 
    df["OppositionHalfPassShare"] = df_safe_div(df["OppositionHalfPasses"], df["Passes"])            
    df["LongBallShare"] = df_safe_div(df["LongBalls"], df["Passes"])                                          
    df["ProgressiveFieldTilt"] = df_safe_div(df["OppositionHalfPasses"], df["OppositionHalfPasses"] + df["OwnHalfPasses"]) 
    df["PassesPerTouch"] = df_safe_div(df["Passes"], df["Touches"])                                
    df["TouchesPerPass"] = df_safe_div(df["Touches"], df["Passes"])                                     
    df["PossessionLossRate"] = df_safe_div(df["PossessionLost"], df["Touches"])                           
    df["UnsuccessfulTouchRate"] = df_safe_div(df["UnsuccessfulTouches"], df["Touches"])                   
    df["DispossessedRate"] = df_safe_div(df["Dispossessed"], df["Touches"])                               

    # Creación
    df["KeyPassesPerPass"] = df_safe_div(df["KeyPasses"], df["Passes"])                                   
    df["AssistConversion"] = df_safe_div(df["GoalAssists"], df["KeyPasses"])                             
    df["ExpectedAssistsPerKeyPass"] = df_safe_div(df["ExpectedAssists"], df["KeyPasses"])               
    df["ExpectedAssistsPerPass"] = df_safe_div(df["ExpectedAssists"], df["Passes"])               
    df["CrossAccuracy"] = df_safe_div(df["AccurateCrosses"], df["Crosses"])                                 
    df["CrossShare"] = df_safe_div(df["Crosses"], df["Passes"])                                              
    df["AccurateCrossShare"] = df_safe_div(df["AccurateCrosses"], df["Passes"])                  
    df["BigChancesPerKeyPass"] = df_safe_div(df["BigChancesCreated"], df["KeyPasses"])   

    # Precisión de pase por zona
    df["PassPercZoneBack"] = df_safe_div(df["PassAccZoneBack"], df["PassZoneBack"])
    df["PassPercZoneCenter"] = df_safe_div(df["PassAccZoneCenter"], df["PassZoneCenter"])
    df["PassPercZoneLeft"] = df_safe_div(df["PassAccZoneLeft"], df["PassZoneLeft"])
    df["PassPercZoneRight"] = df_safe_div(df["PassAccZoneRight"], df["PassZoneRight"])

    # Distribución de pases por zona
    total_zone_passes = (df["PassZoneBack"] + df["PassZoneCenter"] + df["PassZoneLeft"] + df["PassZoneRight"])
    df["PassShareZoneBack"] = df_safe_div(df["PassZoneBack"], total_zone_passes)
    df["PassShareZoneCenter"] = df_safe_div(df["PassZoneCenter"], total_zone_passes)
    df["PassShareZoneLeft"] = df_safe_div(df["PassZoneLeft"], total_zone_passes)
    df["PassShareZoneRight"] = df_safe_div(df["PassZoneRight"], total_zone_passes)

    # Precisión por dirección
    df["PassPercDirBackward"] = df_safe_div(df["PassAccDirBackward"], df["PassDirBackward"])
    df["PassPercDirBackwardLeft"] = df_safe_div(df["PassAccDirBackwardLeft"], df["PassDirBackwardLeft"])
    df["PassPercDirBackwardRight"] = df_safe_div(df["PassAccDirBackwardRight"], df["PassDirBackwardRight"])
    df["PassPercDirForward"] = df_safe_div(df["PassAccDirForward"], df["PassDirForward"])
    df["PassPercDirForwardLeft"] = df_safe_div(df["PassAccDirForwardLeft"], df["PassDirForwardLeft"])
    df["PassPercDirForwardRight"] = df_safe_div(df["PassAccDirForwardRight"], df["PassDirForwardRight"])
    df["PassPercDirLeft"] = df_safe_div(df["PassAccDirLeft"], df["PassDirLeft"])
    df["PassPercDirRight"] = df_safe_div(df["PassAccDirRight"], df["PassDirRight"])

    # Distribución por dirección
    total_dir_passes = (df["PassDirBackward"] + df["PassDirBackwardLeft"] + df["PassDirBackwardRight"] + df["PassDirForward"] + df["PassDirForwardLeft"] + df["PassDirForwardRight"] + df["PassDirLeft"] + df["PassDirRight"])
    df["PassShareDirBackward"] = df_safe_div(df["PassDirBackward"], total_dir_passes)
    df["PassShareDirBackwardLeft"] = df_safe_div(df["PassDirBackwardLeft"], total_dir_passes)
    df["PassShareDirBackwardRight"] = df_safe_div(df["PassDirBackwardRight"], total_dir_passes)
    df["PassShareDirForward"] = df_safe_div(df["PassDirForward"], total_dir_passes)
    df["PassShareDirForwardLeft"] = df_safe_div(df["PassDirForwardLeft"], total_dir_passes)
    df["PassShareDirForwardRight"] = df_safe_div(df["PassDirForwardRight"], total_dir_passes)
    df["PassShareDirLeft"] = df_safe_div(df["PassDirLeft"], total_dir_passes)
    df["PassShareDirRight"] = df_safe_div(df["PassDirRight"], total_dir_passes)
        
    # Finalización
    df["ShotAccuracy"] = df_safe_div(df["ShotsOnTarget"], df["TotalShots"])                                  
    df["GoalConversion"] = df_safe_div(df["Goals"], df["TotalShots"])                                 
    df["BigChanceMissRate"] = df_safe_div(df["BigChancesMissed"], df["BigChancesCreated"])                     
    df["ExpectedGoalsPerShot"] = df_safe_div(df["ExpectedGoals"], df["TotalShots"])                     
    df["ExpectedGoalsOnTargetPerShotOnTarget"] = df_safe_div(df["ExpectedGoalsOnTarget"], df["ShotsOnTarget"])
    df["GoalsMinusxG"] = df["Goals"] - df["ExpectedGoals"]                                               
    df["GoalsMinusxGOT"] = df["Goals"] - df["ExpectedGoalsOnTarget"]                               
    df["BlockedShotRate"] = df_safe_div(df["BlockedShots"], df["TotalShots"])                              
    df["ShotsOnTargetRate"] = df_safe_div(df["ShotsOnTarget"], df["TotalShots"])         
    df["ShotsOffTargetRate"] = df_safe_div(df["ShotsOffTarget"], df["TotalShots"])                     
    df["GoalsPerShotOnTarget"] = df_safe_div(df["Goals"], df["ShotsOnTarget"])                          
    df["HitWoodworkRate"] = df_safe_div(df["HitWoodwork"], df["TotalShots"])        

    # Distribución de resultados
    total_shots = df["MissedShots"] + df["ShotsOnPost"] + df["SavedShots"]
    df["MissedShotRate"] = df_safe_div(df["MissedShots"], total_shots)
    df["PostShotRate"] = df_safe_div(df["ShotsOnPost"], total_shots)
    df["SavedShotRate"] = df_safe_div(df["SavedShots"], total_shots)            

    # Duelos
    df["DuelWinRate"] = df_safe_div(df["DuelsWon"], df["DuelsWon"] + df["DuelsLost"])                      
    df["AerialWinRate"] = df_safe_div(df["AerialWon"], df["AerialWon"] + df["AerialLost"])                
    df["ContestWinRate"] = df_safe_div(df["ContestsWon"], df["Contests"])                                 
    df["ChallengesLostRate"] = df_safe_div(df["ChallengesLost"], df["Contests"])                       

    # Defensa
    df["DefensiveActions"] = df["Tackles"] + df["Interceptions"] + df["Clearances"] + df["OutfielderBlocks"]
    df["DefensiveActionSuccess"] = df_safe_div(df["TacklesWon"] + df["Interceptions"] + df["Clearances"], df["DefensiveActions"])
    df["TackleAccuracy"] = df_safe_div(df["TacklesWon"], df["Tackles"])                                    
    df["InterceptionShare"] = df_safe_div(df["Interceptions"], df["DefensiveActions"])              
    df["ClearanceShare"] = df_safe_div(df["Clearances"], df["DefensiveActions"])                         
    df["BlockShare"] = df_safe_div(df["OutfielderBlocks"], df["DefensiveActions"])                  
    df["BallRecoveryRate"] = df_safe_div(df["BallRecoveries"], df["DefensiveActions"])                 
    df["LastManTackleRate"] = df_safe_div(df["LastManTackles"], df["Tackles"])                          
    df["ClearanceOffLineRate"] = df_safe_div(df["ClearanceOffLine"], df["Clearances"])                       
    df["ErrorsLeadToShotRate"] = df_safe_div(df["ErrorsLeadToShot"], df["DefensiveActions"])         
    df["ErrorsLeadToGoalRate"] = df_safe_div(df["ErrorsLeadToGoal"], df["DefensiveActions"])             
    df["GoalsConcededPerDefAction"] = df_safe_div(df["GoalsConceded"], df["DefensiveActions"])

    # Portero
    df["SaveRate"] = df_safe_div(df["Saves"], df["Saves"] + df["GoalsConceded"])          
    df["SavedShotsInsideBoxRate"] = df_safe_div(df["SavedShotsInsideBox"], df["Saves"])                    
    df["CrossesNotClaimedRate"] = df_safe_div(df["CrossesNotClaimed"], df["CrossesNotClaimed"] + df["HighClaims"])
    df["HighClaimRate"] = df_safe_div(df["HighClaims"], df["HighClaims"] + df["Punches"] + df["CrossesNotClaimed"]) 
    df["PunchRate"] = df_safe_div(df["Punches"], df["HighClaims"] + df["Punches"] + df["CrossesNotClaimed"])     
    df["KeeperSweeperAccuracy"] = df_safe_div(df["AccurateKeeperSweeperActions"], df["KeeperSweeperActions"])    
    df["PenaltySaveRate"] = df_safe_div(df["PenaltySaves"], df["PenaltyFaced"])              
    df["PenaltyGoalConcededRate"] = df_safe_div(df["PenaltyGoalsConceded"], df["PenaltyFaced"])     
    df["GoalsPreventedDiff"] = df["GoalsPrevented"] - df["GoalsConceded"]                         

    # Disciplina
    df["CardsPerFoul"] = df_safe_div(df["YellowCards"] + df["RedCards"], df["Fouls"])                        
    df["YellowCardsPerFoul"] = df_safe_div(df["YellowCards"], df["Fouls"])                                  
    df["RedCardsPerFoul"] = df_safe_div(df["RedCards"], df["Fouls"])                                       
    df["FoulsPerDefAction"] = df_safe_div(df["Fouls"], df["DefensiveActions"])                               
    df["WasFouledRate"] = df_safe_div(df["WasFouled"], df["Touches"])                                         
    df["OffsideRate"] = df_safe_div(df["Offsides"], df["TotalShots"])                                        

    # Balones parados
    df["CornersWonRate"] = df_safe_div(df["CornersWon"], df["Touches"])                                      
    df["CornersLostRate"] = df_safe_div(df["CornersLost"], df["DefensiveActions"])                          
    df["CornersTakenRate"] = df_safe_div(df["CornersTaken"], df["Touches"])                                  
    df["GoalKicksRate"] = df_safe_div(df["GoalKicks"], df["Touches"])                                         

    # Penalitis
    df["PenaltyWonRate"] = df_safe_div(df["PenaltyWon"], df["Touches"])                                      
    df["PenaltyMissRate"] = df_safe_div(df["PenaltyMissed"], df["PenaltyWon"])                              
    df["PenaltyConcededRate"] = df_safe_div(df["PenaltyConceded"], df["DefensiveActions"])     

    # Diferencias
    df["GoalsMinusGoalsConceded"] = df["Goals"] - df["GoalsConceded"]                                    
    df["ShotsMinusGoals"] = df["TotalShots"] - df["Goals"]                                                 
    df["OwnGoalRate"] = df_safe_div(df["OwnGoals"], df["GoalsConceded"])                                 
    
    # Métricas por 90
    cols_exclude = ["Match","Team","Player","ShirtNumber","Position","PassMeanAngle","PassMeanLength","PassMeanX","PassMeanY","ShotMeanX","ShotMeanY","MeanGoalY","MeanGoalZ","ShotMeanLength",
                    "GoalIniX","GoalIniY","GoalFinalY","GoalFinalZ","GoalMeanLength"]
    per90_cols = df.columns.difference(cols_exclude)
    per90_dict = {f"{col}Per90": df[col] * factor_90 for col in per90_cols}
    df = pd.concat([df, pd.DataFrame(per90_dict)], axis=1)

    return df

# Función para la agregación de datos
def player_aggregate_data(player_stats_df: pd.DataFrame) -> pd.DataFrame:

    player_agg = player_stats_df.copy()

    # Sacamos columnas de información
    cols_to_drop = [c for c in ["Match", "Team", "Opponent"] if c in player_agg.columns]
    player_agg = player_agg.drop(columns=cols_to_drop)

    # Sacamos columnas per 90
    cols_per90 = [c for c in player_agg.columns if c.endswith("Per90")]
    player_agg = player_agg.drop(columns=cols_per90)

    ratio_cols = ["Match","Team","PassAccuracy","PassesPerTouch","TouchesPerPass","PossessionLossRate","UnsuccessfulTouchRate","DispossessedRate","OwnHalfPassAccuracy","OwnHalfPassShare",
                  "OppositionHalfPassAccuracy","OppositionHalfPassShare","LongBallAccuracy","LongBallShare","KeyPassesPerPass","AssistConversion","ExpectedAssistsPerKeyPass",
                  "ExpectedAssistsPerPass","CrossAccuracy","CrossShare","AccurateCrossShare","BigChanceMissRate","BigChancesPerKeyPass","ShotAccuracy","ShotsOnTargetRate",
                  "ShotsOffTargetRate","BlockedShotRate","GoalConversion","GoalsPerShotOnTarget","ExpectedGoalsPerShot","ExpectedGoalsOnTargetPerShotOnTarget","HitWoodworkRate",
                  "OffsideRate","DuelWinRate","AerialWinRate","ContestWinRate","ChallengesLostRate","TackleAccuracy","DefensiveActionSuccess","InterceptionShare","ClearanceShare",
                  "BlockShare","BallRecoveryRate","LastManTackleRate","ClearanceOffLineRate","ErrorsLeadToShotRate","ErrorsLeadToGoalRate","GoalsConcededPerDefAction","SaveRate",
                  "SavedShotsInsideBoxRate","CrossesNotClaimedRate","HighClaimRate","PunchRate","KeeperSweeperAccuracy","GoalKicksRate","PenaltySaveRate","PenaltyGoalConcededRate",
                  "CardsPerFoul","YellowCardsPerFoul","RedCardsPerFoul","FoulsPerDefAction","WasFouledRate","PenaltyWonRate","PenaltyMissRate","PenaltyConcededRate","OwnGoalRate",
                  "CornersWonRate","CornersLostRate","CornersTakenRate","GoalsMinusxG","GoalsMinusxGOT","GoalsMinusGoalsConceded","ShotsMinusGoals","PassPercZoneBack",
                  "PassPercZoneCenter","PassPercZoneLeft","PassPercZoneRight","PassPercDirBackward","PassPercDirBackwardLeft","PassPercDirBackwardRight","PassPercDirForward",
                  "PassPercDirForwardLeft","PassPercDirForwardRight","PassPercDirLeft","PassPercDirRight","TotalGoals"]
    ratio_cols = [c for c in ratio_cols if c in player_agg.columns]
    player_agg = player_agg.drop(columns=ratio_cols)

    # Añadimos partidos - para sumar
    player_agg["Matches"] = 1 

    # Diccionario de agregaciones
    agg_dict = {}

    for col in player_agg.columns:
        if col == "Player":
            continue
        elif col in ["PassMeanAngle","PassMeanLength","PassMeanX","PassMeanY","ShotMeanX","ShotMeanY","MeanGoalY","MeanGoalZ","ShotMeanLength","GoalIniX","Elo","OppElo",
                     "GoalIniY","GoalFinalY","GoalFinalZ","GoalMeanLength"]:
            agg_dict[col] = "mean"
        else:
            agg_dict[col] = "sum"

    # Agregación
    player_agg_df = player_agg.groupby("Player", as_index=False).agg(agg_dict)

    # Aplicamos la función de crear nuevas métricas al df
    final_df = player_new_metrics(player_stats_df=player_agg_df)

    return final_df

# Limpieza del dataframe final de agregación de datos
def final_agg_df_cleaning(season_player_df: pd.DataFrame, min_minutes: int = 500) -> pd.DataFrame:

    df = season_player_df.copy()

    # Selección de aquellos jugadores con un minutaje mínimo
    df = df[df["MinutesPlayed"] >= min_minutes]

    # Filtrado de información que queremos sí o sí: si el jugador no tiene fecha de nacimiento, altura o pref foot no lo escogeremos
    df = df.dropna(subset=["DateBirth", "Height", "PrefFoot"])

    # Selección de columnas del dataframe final
    df = df[list_to_order_df]

    # Calculamos edad del jugador
    df["DateBirth"] = pd.to_datetime(df["DateBirth"], format="%d/%m/%Y", errors="coerce")
    today = pd.Timestamp.today()
    df.insert(2, "Age", (today - df["DateBirth"]).dt.days // 365)
    df = df.drop(columns="DateBirth")

    # Rol con la posición al lado para evitar duplicaciones
    df["Role"] = df["Position"] + " - " + df["Role"]

    # Posición general
    df.insert(5, "GeneralPos", np.where(df["Position"].isin(["LW","ST","RW"]), "Attacker", np.where(df["Position"].isin(["LB","CB","RB"]), "Defender", np.where(df["Position"].isin(["DM","AM"]), "Midfielder", "Goalkeeper"))))

    # Realizamos un "Fillna" imputando los valores medios POR POSICIÓN
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df.groupby("Position")[num_cols].transform(lambda x: x.fillna(x.mean()))

    # Aplicamos one-hot-encoding para transformar a numericas las columnas categoricas
    cat_cols = df.select_dtypes(exclude=["number"]).columns.tolist()
    cat_cols = [c for c in cat_cols if c != "Player"]
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)

    return df_encoded.set_index("Player")

# A partir de las funciones creadas anteriormente, crea un dataframe con context y estadisticas de jugadores
def similarity_score_processing(match_info_df: pd.DataFrame, player_stats_df: pd.DataFrame, player_info_df: pd.DataFrame, act_season: str) -> pd.DataFrame:

    # Filtrado de aquellos partidos de la temporada actual - también jugadores que hayan jugado durante aquella temporada
    season_matches = match_info_df[match_info_df["Season"].astype(str) == str(act_season)].reset_index(drop=True)
    season_player_stats = player_stats_df[player_stats_df["Match"].isin(season_matches["ID"].unique().tolist())].reset_index(drop=True)
    season_player_info = player_info_df[player_info_df["ID"].isin(player_stats_df["Player"].unique().tolist())].reset_index(drop=True)

    # Seleccionamos solo las columnas que necesitaremos de información
    season_player_info = season_player_info[["ID","DateBirth","Height","PrefFoot","Position","Role"]]
    season_player_stats = season_player_stats.drop(columns=["ShirtNumber", "Position"])
    season_matches = season_matches[["ID","HomeTeam","AwayTeam","HomeScore","AwayScore","HomeElo","AwayElo"]]

    # Cambio de índice en el dataframe de partido - una fila por equipo y partido
    home_season_matches = season_matches.rename(columns={"ID":"Match","HomeTeam":"Team","AwayTeam":"Opponent","HomeScore":"Score","AwayScore":"OppScore","HomeElo":"Elo","AwayElo":"OppElo"})
    away_season_matches = season_matches.rename(columns={"ID":"Match","AwayTeam":"Team","HomeTeam":"Opponent","AwayScore":"Score","HomeScore":"OppScore","AwayElo":"Elo","HomeElo":"OppElo"})
    season_matches = pd.concat([home_season_matches, away_season_matches]).sort_values(by=["Match", "Team"]).reset_index(drop=True)

    # Concatenamos la información del partido con las estadísticas del jugador para obtener información extra
    season_matches_stats = season_player_stats.merge(season_matches, on=["Match", "Team"], how="left")

    # Aplicamos agregación de datos
    season_player_agg_stats = player_aggregate_data(player_stats_df=season_matches_stats)

    # Merge de los datos agregados de la temporada con la información del jugador
    season_player_info = season_player_info.rename(columns={"ID":"Player"})
    season_player_df = season_player_info.merge(season_player_agg_stats, how="inner", on="Player")

    # Limpieza del dataframe
    final_player_df = final_agg_df_cleaning(season_player_df=season_player_df)

    return final_player_df

In [14]:
# Aplicado de la función de similaridad
proc_df = similarity_score_processing(match_info_df=match_info_df.copy(), player_stats_df=player_stats_df.copy(), player_info_df=player_info_df.copy(), act_season=ACT_SEASON)

display(proc_df.head())

C:\Users\ASUS\AppData\Local\Temp\ipykernel_5424\2045756652.py:30: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  result = np.where((den != 0) & (~pd.isna(num)) & (~pd.isna(den)), num / den, np.nan)


,Age,Height,MinutesPlayed,Matches,Elo,OppElo,Score,OppScore,TouchesPer90,PassesPer90,...,Role_DM - Deep Playmaker,Role_FB - Balanced,Role_FB - Defensive,Role_FB - Runner,Role_GK - Direct Outlet,Role_ST - Linker,Role_ST - Poacher,Role_WG - Explosive,Role_WG - Finisher,Role_WG - Support
Player,,,,,,,,,,,,,,,,,,,,,
L8U04,20,190.0,785.0,13,1650.886946,1654.628242,25.0,22.0,64.6344,50.0802,...,0,0,0,0,0,0,0,0,0,0
ERHSS,28,178.0,1301.0,25,1372.030528,1461.065511,29.0,48.0,58.9584,49.8240,...,0,0,0,0,0,0,0,0,0,0
KVXR9,21,173.0,1622.0,23,1629.339642,1626.626804,30.0,35.0,68.4315,50.7825,...,1,0,0,0,0,0,0,0,0,0
93Y47,31,192.0,720.0,8,971.716186,1289.194111,8.0,14.0,32.6250,23.8750,...,0,0,0,0,1,0,0,0,0,0
CX0W5,18,183.0,587.0,15,1419.316241,1462.199137,18.0,21.0,38.0184,22.8417,...,0,0,0,0,0,0,0,0,0,0


---

## **2. Cómputo de la matriz de similitud**

Las similitudes solo tienen sentido **entre jugadores comparables**, así que se calcula una matriz separada para cada **grupo de posición general**: porteros, defensas, centrocampistas y delanteros.

Para cada grupo:

1. Se filtran los jugadores del grupo y se eliminan columnas sin variabilidad o con nulos.
2. Se **estandarizan** las variables (z-score con `StandardScaler`) para que todas pesen por igual.
3. Se calcula la matriz de ***cosine similarity***. Para dos jugadores con vectores de estadísticas $\mathbf{a}$ y $\mathbf{b}$:

$$
\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert}
= \frac{\sum_{i} a_i b_i}{\sqrt{\sum_i a_i^2}\,\sqrt{\sum_i b_i^2}}
$$

El resultado está en $[-1, 1]$: **1** = patrones idénticos, **0** = sin relación, **−1** = opuestos.

> *Nota:* la función soporta también distancia euclídea como alternativa, convertida a similitud mediante $1 / (1 + d)$, y pesos opcionales por columna; por defecto se usa el coseno con todas las variables igual de ponderadas.

### **2.1. Conversión a porcentaje**

Para una lectura más intuitiva, la similitud se reescala de $[-1, 1]$ a un **porcentaje** $[0, 100]$:

$$
\text{Similitud (\%)} = \frac{\cos(\mathbf{a}, \mathbf{b}) + 1}{2} \cdot 100
$$

In [15]:
# Cómputo de la matrix de similaridad a partir del dataframe de jugadores
def compute_similarity_matrix(df: pd.DataFrame, metric: str ="cosine", scale: bool =True, weights: dict = None) -> pd.DataFrame:

    X = df.copy()

    # Pesos opcionales
    if weights is not None:
        for col, w in weights.items():
            if col in X.columns:
                X[col] = X[col] * w

    # Escalado de metricas
    if scale:
        scaler = StandardScaler()
        X_values = scaler.fit_transform(X)
    else:
        X_values = X.values

    # Matriz de similaridad
    if metric == "cosine":
        sim_matrix = cosine_similarity(X_values)
    elif metric == "euclidean":
        dist_matrix = euclidean_distances(X_values)
        sim_matrix = 1 / (1 + dist_matrix)

    # Convertimos a dataframe
    sim_df = pd.DataFrame(sim_matrix, index=df.index, columns=df.index)

    return sim_df

# Función para la preparación de todos los datos
def sim_matrix_proc(proc_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]: 

    final_player_df = proc_df.copy()

    # Aplicado de matriz de similitud en porteros
    only_goalkeepers = final_player_df[final_player_df["GeneralPos_Goalkeeper"] == 1]
    only_goalkeepers = only_goalkeepers.loc[:, only_goalkeepers.nunique(dropna=False) > 1]          # Eliminamos columnas sin variabilidad
    only_goalkeepers = only_goalkeepers.dropna(axis=1)
    sim_matrix_goalkeepers = compute_similarity_matrix(df=only_goalkeepers)
    sim_matrix_goalkeepers.index.name = None
    sim_matrix_goalkeepers.columns.name = None

    # Aplicado de matriz de similitud en defensas
    only_defenders = final_player_df[final_player_df["GeneralPos_Defender"] == 1]
    only_defenders = only_defenders.loc[:, only_defenders.nunique(dropna=False) > 1]                # Eliminamos columnas sin variabilidad
    only_defenders = only_defenders.dropna(axis=1)
    sim_matrix_defenders = compute_similarity_matrix(df=only_defenders)
    sim_matrix_defenders.index.name = None
    sim_matrix_defenders.columns.name = None

    # Aplicado de matriz de similitud en centrocampistas
    only_midfielders = final_player_df[final_player_df["GeneralPos_Midfielder"] == 1]
    only_midfielders = only_midfielders.loc[:, only_midfielders.nunique(dropna=False) > 1]          # Eliminamos columnas sin variabilidad
    only_midfielders = only_midfielders.dropna(axis=1)
    sim_matrix_midfielders = compute_similarity_matrix(df=only_midfielders)
    sim_matrix_midfielders.index.name = None
    sim_matrix_midfielders.columns.name = None

    # Aplicado de matriz de similitud en centrocampistas
    only_forward = final_player_df[final_player_df["GeneralPos_Attacker"] == 1]
    only_forward = only_forward.loc[:, only_forward.nunique(dropna=False) > 1]                      # Eliminamos columnas sin variabilidad
    only_forward = only_forward.dropna(axis=1)
    sim_matrix_attackers = compute_similarity_matrix(df=only_forward)
    sim_matrix_attackers.index.name = None
    sim_matrix_attackers.columns.name = None

    return sim_matrix_goalkeepers, sim_matrix_defenders, sim_matrix_midfielders, sim_matrix_attackers

In [16]:
# Aplicamos la función para obtener las matrices de similitud entre pares de jugadores
sim_matrix_goalkeepers, sim_matrix_defenders, sim_matrix_midfielders, sim_matrix_attackers = sim_matrix_proc(proc_df=proc_df.copy())

# Las matrices de similitud tienen valores de -1 a 1, vamos a extrapolarlo a porcentage (0 a 100)
corr_perc_goalkeepers = ((sim_matrix_goalkeepers + 1) / 2) * 100
corr_perc_defenders = ((sim_matrix_defenders + 1) / 2) * 100
corr_perc_midfielders = ((sim_matrix_midfielders + 1) / 2) * 100
corr_perc_attackers = ((sim_matrix_attackers + 1) / 2) * 100

display(corr_perc_defenders.head())

,L8U04,MM2XV,X46KF,QAWAK,6X8MU,IMDAD,4INJ7,XN5YI,G7XGC,GGRDF,...,NDSLJ,GOG3F,ON8NU,DDLBM,B0FAV,6TTA8,G1Q48,1G3DN,JVQDP,IAHQU
L8U04,100.000000,51.477065,52.859446,42.582960,52.756382,47.570836,57.682706,46.899298,36.244130,48.820961,...,40.539929,44.411215,48.274003,56.460492,54.863010,55.595257,52.002441,50.391976,55.212786,48.366623
MM2XV,51.477065,100.000000,55.416346,53.328496,64.996791,70.113717,69.594765,69.119812,67.726713,65.179173,...,66.569213,45.529851,50.062040,54.334127,40.549077,38.785165,48.703161,57.629380,68.810259,50.682092
X46KF,52.859446,55.416346,100.000000,38.999155,50.342357,57.221861,57.377115,52.772361,42.236942,61.202089,...,65.815994,48.575167,47.224271,44.235668,46.544447,41.827667,54.877252,37.779281,48.560618,48.678384
QAWAK,42.582960,53.328496,38.999155,100.000000,49.843565,45.146803,54.308601,44.125359,51.120100,43.002754,...,56.827786,45.929008,51.424572,63.952843,57.496390,49.544514,56.608973,59.595183,49.449590,42.743264
6X8MU,52.756382,64.996791,50.342357,49.843565,100.000000,64.654996,61.333583,55.103768,59.936210,55.332748,...,60.775677,43.180338,49.000231,49.996005,51.579321,48.510261,52.943068,49.859295,48.745935,66.751751


---

## **3. Top de jugadores más similares**

Se define una función que, dado el ID de un jugador:

1. Detecta en qué matriz (grupo de posición) está.
2. Toma su columna de similitudes y **elimina al propio jugador**.
3. Ordena de mayor a menor y devuelve el **top N** con su porcentaje.

Por ejemplo, para Pau Cubarsí se obtiene la lista de los 10 centrales más parecidos en estilo, con su porcentaje de similitud.

In [17]:
# Función para obtener el top n de jugadores más parecidos al jugador definido
def get_similar_players(player_id: str, top_n: int, corr_perc_goalkeepers: pd.DataFrame, corr_perc_defenders: pd.DataFrame, corr_perc_midfielders: pd.DataFrame, corr_perc_attackers: pd.DataFrame) -> dict:

    # Comprovamos en que posición predomina el jugador
    if player_id in corr_perc_goalkeepers.columns:
        sim_df = corr_perc_goalkeepers
    elif player_id in corr_perc_defenders.columns:
        sim_df = corr_perc_defenders
    elif player_id in corr_perc_midfielders.columns:
        sim_df = corr_perc_midfielders
    elif player_id in corr_perc_attackers.columns:
        sim_df = corr_perc_attackers

    # Serie de similitudes
    sim_series = sim_df[player_id].copy()
    sim_series = sim_series.drop(player_id)         # Eliminamos el propio jugador

    # Ordenar de mayor a menor y selección de top N
    sim_series = sim_series.sort_values(ascending=False)
    top_similar = sim_series.head(top_n)                    

    return top_similar.to_dict()

In [25]:
# Ejemplo - Pau Cubarsí
similar_players = get_similar_players(player_id="2F5HQ", top_n=10, corr_perc_goalkeepers=corr_perc_goalkeepers, corr_perc_defenders=corr_perc_defenders, corr_perc_midfielders=corr_perc_midfielders, corr_perc_attackers=corr_perc_attackers)

players_names_dict = dict(zip(player_info_df["ID"], player_info_df["Name"]))
i = 1
for player, similarity in similar_players.items():
    print(f"{i} - {players_names_dict.get(player)} similarity with Pau Cubarsí -> {round(similarity, 2)}%")
    i += 1

1 - Willian Pacho similarity with Pau Cubarsí -> 81.86%
2 - Shogo Taniguchi similarity with Pau Cubarsí -> 81.1%
3 - Alexandre Penetra similarity with Pau Cubarsí -> 80.22%
4 - Pantelis Hatzidiakos similarity with Pau Cubarsí -> 80.12%
5 - Manuel Akanji similarity with Pau Cubarsí -> 79.58%
6 - Trevoh Chalobah similarity with Pau Cubarsí -> 79.3%
7 - Rúben Dias similarity with Pau Cubarsí -> 78.67%
8 - Jan Bednarek similarity with Pau Cubarsí -> 77.87%
9 - Raúl Asencio similarity with Pau Cubarsí -> 77.4%
10 - Gerard Martín similarity with Pau Cubarsí -> 77.32%


---